<a href="https://colab.research.google.com/github/alwhshalkasr267-design/FinalProject/blob/main/FinalProject_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import pandas as pd
BASE_DIR = "/content/drive/MyDrive/obstacles_dataset"
Chair_DIR = os.path.join(BASE_DIR, "chair")
Door_DIR = os.path.join(BASE_DIR, "door")
Fence_DIR = os.path.join(BASE_DIR, "fence")
Garbage_bin_DIR = os.path.join(BASE_DIR, "garbage_bin")
Obstacle_DIR = os.path.join(BASE_DIR, "obstacle")
Plant_DIR = os.path.join(BASE_DIR, "plant")
Pothole_DIR = os.path.join(BASE_DIR, "pothole")
Stairs_DIR = os.path.join(BASE_DIR, "stairs")
Table_DIR = os.path.join(BASE_DIR, "table")
Vehicle_DIR = os.path.join(BASE_DIR, "vehicle")


In [8]:
Chair_count = len(os.listdir(Chair_DIR))
Door_count = len(os.listdir(Door_DIR))
Fence_count = len(os.listdir(Fence_DIR))
Garbage_bin_count = len(os.listdir(Garbage_bin_DIR))
Obstacle_count = len(os.listdir(Obstacle_DIR))
Plant_count = len(os.listdir(Plant_DIR))
Pothole_count = len(os.listdir(Pothole_DIR))
Stairs_count = len(os.listdir(Stairs_DIR))
Table_count = len(os.listdir(Table_DIR))
Vehicle_count = len(os.listdir(Vehicle_DIR))


print ("Number of Chair images: ", Chair_count)
print ("Number of Door images: ", Door_count)
print ("Number of Fence images: ", Fence_count)
print ("Number of Garbage_bin images: ", Garbage_bin_count)
print ("Number of Obstacle images: ", Obstacle_count)
print ("Number of Plant images: ", Plant_count)
print ("Number of Pothole images: ", Pothole_count)
print ("Number of Stairs images: ", Stairs_count)
print ("Number of Table images: ", Table_count)
print ("Number of Vehicle images: ", Vehicle_count)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/obstacles_dataset/chair'

In [ ]:
images = []
labels = []

for file in os.listdir(Chair_DIR):
    images.append(os.path.join(Chair_DIR, file))
    labels.append("0") # Chair

for file in os.listdir(Door_DIR):
    images.append(os.path.join(Door_DIR, file))
    labels.append("1") # Door

for file in os.listdir(Fence_DIR):
    images.append(os.path.join(Fence_DIR, file))
    labels.append("2") # Fence

for file in os.listdir(Garbage_bin_DIR):
    images.append(os.path.join(Garbage_bin_DIR, file))
    labels.append("3") # Garbage_bin

for file in os.listdir(Obstacle_DIR):
    images.append(os.path.join(Obstacle_DIR, file))
    labels.append("4") # Obstacle

for file in os.listdir(Plant_DIR):
    images.append(os.path.join(Plant_DIR, file))
    labels.append("5") # Plant

for file in os.listdir(Pothole_DIR):
    images.append(os.path.join(Pothole_DIR, file))
    labels.append("6") # Pothole

for file in os.listdir(Stairs_DIR):
    images.append(os.path.join(Stairs_DIR, file))
    labels.append("7") # Stairs

for file in os.listdir(Table_DIR):
    images.append(os.path.join(Table_DIR, file))
    labels.append("8") # Table

for file in os.listdir(Vehicle_DIR):
    images.append(os.path.join(Vehicle_DIR, file))
    labels.append("9") # Vehicle

df = pd.DataFrame({
"image" : images ,
"label" : labels
})
df = df.sample(frac=1).reset_index(drop=True) # Mixing up the raws
df.head(20)


,image,label
0,/content/drive/MyDrive/obstacles dataset/stair...,7
1,/content/drive/MyDrive/obstacles dataset/potho...,6
2,/content/drive/MyDrive/obstacles dataset/obsta...,4
3,/content/drive/MyDrive/obstacles dataset/garba...,3
4,/content/drive/MyDrive/obstacles dataset/garba...,3
5,/content/drive/MyDrive/obstacles dataset/door/...,1
6,/content/drive/MyDrive/obstacles dataset/vehic...,9
7,/content/drive/MyDrive/obstacles dataset/stair...,7
8,/content/drive/MyDrive/obstacles dataset/chair...,0
9,/content/drive/MyDrive/obstacles dataset/obsta...,4


In [ ]:
from PIL import Image
from tqdm import tqdm

bad = []

for p in tqdm(df['image']):
  try:
    with Image.open(p) as im:
      im.load()
  except:
        bad.append(p)

print("Bad images: ", len(bad))

df = df[~df['image'].isin(bad)].reset_index(drop=True)
print("Remaining images: ",len(df))

100%|██████████| 3678/3678 [17:35<00:00,  3.48it/s]

Bad images:  0
Remaining images:  3678


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

data_dir = BASE_DIR
train_dir = "dataset/train"
val_dir = "dataset/val"
test_dir = "dataset/test"

val_ratio = 0.15
test_ratio = 0.15

classes = os.listdir(data_dir)

for cls in classes:
    src_cls_dir = os.path.join(data_dir, cls)
    if not os.path.isdir(src_cls_dir):
        continue

    images = os.listdir(src_cls_dir)

    train_val_images, test_images = train_test_split(images, test_size=test_ratio, random_state=42)

    relative_val_ratio = val_ratio /(1-test_ratio)
    train_images, val_images = train_test_split(
         train_val_images, test_size=relative_val_ratio,random_state=42
    )

    for split_dir , split_images in zip(
        [train_dir, val_dir, test_dir], [train_images, val_images, test_images]):
      dset_dir = os.path.join(split_dir, cls)
      os.makedirs(dset_dir, exist_ok=True)

      for img in split_images:
        shutil.copy(os.path.join(src_cls_dir, img), os.path.join(dset_dir, img))

print("تم تقسيم البيانات بنجاح ")


تم تقسيم البيانات بنجاح 


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. تحديد حجم الصور الموحد وحجم الدفعة (Batch Size)
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# 2. عمل جيل للمخرجات وعمل التطبيع (Rescaling) للبكسلات بين 0 و 1
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,      # عمل دوران خفيف للصور لزيادة دقة التدريب
    zoom_range=0.15,        # زووم خفيف
    horizontal_flip=True    # قلب الصور أفقياً
)

# بالنسبة لبيانات التقييم والاختبار نعمل فقط تطبيع بدون تلاعب بالصور
val_test_datagen = ImageDataGenerator(rescale=1./255)

# 3. ربط المولدات بالمجلدات التي أنشأتها في الخطوة السابقة
print("Loading Training Set:")
train_generator = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical' # لأن عندك 11 فئة (أكثر من فئتين)
)

print("\nLoading Validation Set:")
val_generator = val_test_datagen.flow_from_directory(
    'dataset/val',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("\nLoading Test Set:")
test_generator = val_test_datagen.flow_from_directory(
    'dataset/test',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # لا نريد خلط بيانات الاختبار لنستخرج تقرير دقيق لاحقاً
)

Loading Training Set:
Found 2567 images belonging to 10 classes.

Loading Validation Set:
Found 555 images belonging to 10 classes.

Loading Test Set:
Found 556 images belonging to 10 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. تحميل النموذج المجهز مسبقاً من جوجل بدون الطبقة الأخيرة
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# تجميد أوزان النموذج الأساسي حتى لا تتغير أثناء التدريب البدئي
base_model.trainable = False

# 2. بناء الهيكل الخاص بمشروعك (المصنف المخصص لـ 11 فئة)
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3), # للحماية من الإفراط في التخصيص (Overfitting)
    layers.Dense(11, activation='softmax') # الرقم 11 يمثل عدد الفئات لديك
])

# 3. تجميع النموذج (Compile) وتحديد دالة الخسارة والمحسن
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# طباعة ملخص هيكل النموذج
model.summary()

# 4. بدء عملية التدريب (Training)
EPOCHS = 10 # يمكنك زيادة العدد لاحقاً (مثلاً إلى 20) لتحسين الدقة
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,371 (9.24 MB)

 Trainable params: 165,387 (646.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 10), output.shape=(None, 11)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 1. تقييم النموذج على بيانات الاختبار واستخراج الدقة الإجمالية
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nTest Accuracy (الدقة على بيانات الاختبار): {test_acc * 100:.2f}%")

# 2. عمل توقعات لمعرفة الفئات المحددة لكل صورة
predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# 3. طباعة تقرير تصنيف مفصل (يظهر الدقة لكل فئة من الـ 11 فئة بشكل منفصل)
print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

# 4. حفظ النموذج بالكامل في ملف جاهز
model.save("my_obstacles_model.h5")
print("\nتم حفظ النموذج بنجاح باسم: my_obstacles_model.h5")

ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 10), output.shape=(None, 11)